<a href="https://colab.research.google.com/github/isabelle-sauget/democracy-econ_growth_analysis/blob/main/democracy_econ_growth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pregatirea dataframe-ului


In [1]:
from google.colab import data_table
data_table.enable_dataframe_formatter()

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import math


In [3]:
df0 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/V-Dem-CY-Full+Others-v16.csv",
                  low_memory = False)

In [ ]:
print(df0.head())

  country_name country_text_id  country_id  year historical_date  project  \
0       Mexico             MEX           3  1789      1789-12-31        1   
1       Mexico             MEX           3  1790      1790-12-31        1   
2       Mexico             MEX           3  1791      1791-12-31        1   
3       Mexico             MEX           3  1792      1792-12-31        1   
4       Mexico             MEX           3  1793      1793-12-31        1   

   historical                  histname  codingstart  codingend  ...  \
0           1  Viceroyalty of New Spain         1789       2025  ...   
1           1  Viceroyalty of New Spain         1789       2025  ...   
2           1  Viceroyalty of New Spain         1789       2025  ...   
3           1  Viceroyalty of New Spain         1789       2025  ...   
4           1  Viceroyalty of New Spain         1789       2025  ...   

   e_pechmor  e_miinteco  e_civil_war  e_miinterc  e_pt_coup  \
0        NaN         0.0          NaN   

In [ ]:
#caut cum se numesc in baza de date variabilele pentru care am primit KeyError
gasite_1 = [col for col in df0.columns if 'v2lgqugen' in col]
gasite_2 = [col for col in df0.columns if 'v2petersch' in col]
gasite_3 = [col for col in df0.columns if 'v2smhargr' in col]

print("Parlament:", gasite_1)
print("Educatie:", gasite_2)
print("Hartuire Online:", gasite_3)

#variabila pentru hartuire este o variabila dummy si are 12 variante deci o elimin din analiza

Parlament: ['v2lgqugen', 'v2lgqugens', 'v2lgqugent']
Educatie: ['v2petersch']
Hartuire Online: ['v2smhargr_0', 'v2smhargr_1', 'v2smhargr_2', 'v2smhargr_3', 'v2smhargr_4', 'v2smhargr_5', 'v2smhargr_6', 'v2smhargr_7', 'v2smhargr_8', 'v2smhargr_9', 'v2smhargr_10', 'v2smhargr_nr']


In [6]:
with open("/content/drive/MyDrive/Colab Notebooks/columns.txt", "r") as file:
  de_pastrat = [line.strip() for line in file if line.strip()]

coloane = ['country_name','country_text_id','year'] + de_pastrat
df = df0[coloane].copy()

In [7]:
print(df.shape, "\n")
print(df.info())

(28092, 45) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28092 entries, 0 to 28091
Data columns (total 45 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   country_name        28092 non-null  object 
 1   country_text_id     28092 non-null  object 
 2   year                28092 non-null  int64  
 3   v2elintim_osp       15984 non-null  float64
 4   v2elvotbuy_osp      15982 non-null  float64
 5   v2elfrfair_osp      15982 non-null  float64
 6   v2elaccept_osp      15933 non-null  float64
 7   v2elasmoff_osp      15980 non-null  float64
 8   v2elffelr_osp       18388 non-null  float64
 9   v2psbars_osp        27422 non-null  float64
 10  v2psoppaut_osp      23444 non-null  float64
 11  v2exrescon_osp      26522 non-null  float64
 12  v2exbribe_osp       27493 non-null  float64
 13  v2exembez_osp       27396 non-null  float64
 14  v2lgotovst_osp      22202 non-null  float64
 15  v2lgqugen           18957 non-null  flo

In [26]:
#data_table.DataTable(df, max_columns=50, max_rows=30000)

In [27]:
#df[df['year'] == 2019][['country_name','year','e_gdppc']]

# Alipirea cu datele PIB per capita (PPP 2021) de pe World Bank


In [16]:
an = '2024'
df_wb0 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/GDPPC.csv", skiprows=4)
#setul de date World Bank contine cate o coloana pentru FIECARE an,
#spre deosebire de cel V-Dem cu o singura coloana INT cu denumirea 'year'
#pentru World Bank, anul este STRING
df_wb = df_wb0[['Country Code', an]].copy()
df_wb.columns = ['ISO_code', "GDPPC"]
print(df_wb)

    ISO_code         GDPPC
0        ABW  44558.928759
1        AFE   4106.434140
2        AFG           NaN
3        AFW   5996.395736
4        AGO   8901.887366
..       ...           ...
261      XKX  15716.292556
262      YEM           NaN
263      ZAF  13597.653850
264      ZMB   3708.069043
265      ZWE   5215.253011

[266 rows x 2 columns]


In [17]:
df_vdem_perAn = df[df['year'] == int(an)].copy()

In [18]:
df_final = pd.merge(df_vdem_perAn, df_wb,
                    left_on='country_text_id',
                    right_on="ISO_code",
                    how="inner")

print(f"Setul de date are {df_final.shape[0]} tari si {df_final.shape[1]-4} variabile")
#excluzand e_gdppc al Maddison Project cu valoare lipsa pentru 2024
display(df_final.head())

Setul de date are 175 tari si 43 variabile


,country_name,country_text_id,year,v2elintim_osp,v2elvotbuy_osp,v2elfrfair_osp,v2elaccept_osp,v2elasmoff_osp,v2elffelr_osp,v2psbars_osp,...,v2caautmob_osp,v2capolit_osp,v2cacritic_osp,v2smgovcapsec_osp,v2smpardom_osp,v2smfordom_osp,v2smpolsoc_osp,e_gdppc,ISO_code,GDPPC
0,Mexico,MEX,2024,2.661,1.773,3.004,3.907,1.999,2.979,3.541,...,0.615,1.403,2.829,2.180,0.816,2.981,0.332,NaN,MEX,22039.628702
1,Suriname,SUR,2024,3.641,2.017,3.792,3.764,1.998,3.449,3.873,...,0.096,2.103,1.346,1.565,2.447,3.030,1.187,NaN,SUR,19179.101832
2,Sweden,SWE,2024,3.918,3.808,3.864,3.895,1.999,3.947,3.960,...,0.018,2.002,3.418,3.132,3.192,2.786,1.751,NaN,SWE,62978.939046
3,Switzerland,CHE,2024,3.883,3.750,3.803,3.919,1.999,3.870,3.947,...,0.043,2.618,2.698,3.051,2.994,3.896,0.604,NaN,CHE,82286.180190
4,Ghana,GHA,2024,3.437,1.520,3.177,3.684,1.998,2.927,3.914,...,0.041,2.529,3.632,1.639,2.093,3.460,0.446,NaN,GHA,7055.624960


In [14]:
print("randuri v-dem", len(df_vdem_perAn))
print(df_vdem_perAn['country_text_id'].head(3).tolist())
print("randuri wb", len(df_wb))
print(df_wb['ISO_code'].head(3).tolist())

randuri v-dem 179
['MEX', 'SUR', 'SWE']
randuri wb 266
['ABW', 'AFE', 'AFG']


In [ ]:
#df_final = df_final.dropna().copy()

In [19]:
# isnull().sum() computeaza suma valorilor lipsa PER fiecare coloana
print(df_final.isnull().sum())
#sterg acum coloana nula cu PIB a setului de date V-Dem
#si coloana tertiary school enrolment pentru ca are toate valorile LIPSA
df_final.drop(columns=['e_gdppc','v2petersch'], inplace=True)
print(df_final.isnull().sum())


country_name            0
country_text_id         0
year                    0
v2elintim_osp          17
v2elvotbuy_osp         17
v2elfrfair_osp         17
v2elaccept_osp         17
v2elasmoff_osp         17
v2elffelr_osp          22
v2psbars_osp            0
v2psoppaut_osp          7
v2exrescon_osp          0
v2exbribe_osp           0
v2exembez_osp           0
v2lgotovst_osp          3
v2lgqugen               3
v2dlcountr_osp          0
v2juncind_osp           0
v2juhcind_osp           0
v2jucorrdc_osp          0
v2jucomp_osp            0
v2clrspct_osp           0
v2clacjstw_osp          0
v2pepwrgen_osp          0
v2stcritrecadm_osp      0
v2cseeorgs_osp          0
v2csantimv_osp          0
v2csrlgcon_osp          0
v2mecenefi_osp          0
v2mebias_osp            0
v2mecorrpt_osp          0
v2pepwrort_osp          0
v2pepwrses_osp          0
v2pehealth_osp          0
v2petersch            175
v2cacamps_osp           1
v2caviol_osp            1
v2caautmob_osp          1
v2capolit_os

In [20]:
#analiza valorilor lipsa si alegerea variabilelor pe care sa le pastrez
print( ((df_final.isna().sum())/len(df_final))*100 )

country_name           0.000000
country_text_id        0.000000
year                   0.000000
v2elintim_osp          9.714286
v2elvotbuy_osp         9.714286
v2elfrfair_osp         9.714286
v2elaccept_osp         9.714286
v2elasmoff_osp         9.714286
v2elffelr_osp         12.571429
v2psbars_osp           0.000000
v2psoppaut_osp         4.000000
v2exrescon_osp         0.000000
v2exbribe_osp          0.000000
v2exembez_osp          0.000000
v2lgotovst_osp         1.714286
v2lgqugen              1.714286
v2dlcountr_osp         0.000000
v2juncind_osp          0.000000
v2juhcind_osp          0.000000
v2jucorrdc_osp         0.000000
v2jucomp_osp           0.000000
v2clrspct_osp          0.000000
v2clacjstw_osp         0.000000
v2pepwrgen_osp         0.000000
v2stcritrecadm_osp     0.000000
v2cseeorgs_osp         0.000000
v2csantimv_osp         0.000000
v2csrlgcon_osp         0.000000
v2mecenefi_osp         0.000000
v2mebias_osp           0.000000
v2mecorrpt_osp         0.000000
v2pepwro

In [21]:
iso = df_final.pop('ISO_code')
df_final = df_final.drop(columns='country_text_id')
df_final.insert(1,'iso_code', iso)
print(df_final.head())

  country_name iso_code  year  v2elintim_osp  v2elvotbuy_osp  v2elfrfair_osp  \
0       Mexico      MEX  2024          2.661           1.773           3.004   
1     Suriname      SUR  2024          3.641           2.017           3.792   
2       Sweden      SWE  2024          3.918           3.808           3.864   
3  Switzerland      CHE  2024          3.883           3.750           3.803   
4        Ghana      GHA  2024          3.437           1.520           3.177   

   v2elaccept_osp  v2elasmoff_osp  v2elffelr_osp  v2psbars_osp  ...  \
0           3.907           1.999          2.979         3.541  ...   
1           3.764           1.998          3.449         3.873  ...   
2           3.895           1.999          3.947         3.960  ...   
3           3.919           1.999          3.870         3.947  ...   
4           3.684           1.998          2.927         3.914  ...   

   v2cacamps_osp  v2caviol_osp  v2caautmob_osp  v2capolit_osp  v2cacritic_osp  \
0          

In [28]:
print(df_final.shape)
#data_table.DataTable(df_final, max_columns=50, max_rows=200)

(175, 39)


In [23]:
#per rand, deci per tara
tari_cu_na = df_final[df_final.isna().any(axis=1)]
print(len(tari_cu_na), "de tari cu date lipsa din",df_final.shape[0])

for index, row in tari_cu_na.iterrows():
  tara = row['country_name']
  variabile_lipsa = row[row.isna()].index.tolist()
  print(f"Pentru {tara} lipsesc: {variabile_lipsa}\n")

43 de tari cu date lipsa din 175
Pentru Burma/Myanmar lipsesc: ['v2elintim_osp', 'v2elvotbuy_osp', 'v2elfrfair_osp', 'v2elaccept_osp', 'v2elasmoff_osp', 'v2elffelr_osp', 'v2lgotovst_osp']

Pentru Egypt lipsesc: ['v2elffelr_osp']

Pentru Yemen lipsesc: ['v2elintim_osp', 'v2elvotbuy_osp', 'v2elfrfair_osp', 'v2elaccept_osp', 'v2elasmoff_osp', 'v2elffelr_osp', 'GDPPC']

Pentru Bangladesh lipsesc: ['v2lgotovst_osp', 'v2lgqugen']

Pentru Bolivia lipsesc: ['v2elffelr_osp']

Pentru Haiti lipsesc: ['v2elintim_osp', 'v2elvotbuy_osp', 'v2elfrfair_osp', 'v2elaccept_osp', 'v2elasmoff_osp']

Pentru Mali lipsesc: ['v2elintim_osp', 'v2elvotbuy_osp', 'v2elfrfair_osp', 'v2elaccept_osp', 'v2elasmoff_osp', 'v2elffelr_osp']

Pentru South Sudan lipsesc: ['v2elintim_osp', 'v2elvotbuy_osp', 'v2elfrfair_osp', 'v2elaccept_osp', 'v2elasmoff_osp', 'v2elffelr_osp', 'GDPPC']

Pentru Sudan lipsesc: ['v2elintim_osp', 'v2elvotbuy_osp', 'v2elfrfair_osp', 'v2elaccept_osp', 'v2elasmoff_osp', 'v2lgotovst_osp']

Pentru Vie

In [24]:
# din start, va trebui sa elimin tarile care nu au PIB raportat
# multor tari le lipseste v2elffelr_osp, adica valori pentru indicatorul Alegerilor Locale, motiv pentru care voi elimina variabila din
# dataset (exista si alti indicatori similari acestuia, deci pot elimina)
#ELIMIN variabilele care au NA pentru China, deoarece prioritizez pastrarea Chinei in analiza

df_final.drop(columns=['v2elintim_osp', 'v2elvotbuy_osp', 'v2elfrfair_osp', 'v2elaccept_osp', 'v2elasmoff_osp'], inplace=True)


In [25]:
cale_fisier = "/content/drive/MyDrive/Colab Notebooks/dataframe_2024.csv"
df_final.to_csv(cale_fisier, index=False)